# Pipeline Unificado com ColumnTransformer no Dataset California Housing

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Carregamento do dataset original
california = fetch_california_housing(as_frame=True)
X = california.data.copy()
y = california.target

# Criando uma coluna artificial para mostrar o One-Hot Encoding
np.random.seed(42)
X['region'] = np.random.choice(['North', 'South', 'East', 'West'], size=len(X))

# Divisao em treino e teste para garantir a prevençao de vazamento de dados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

display(X_train.head())

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,region
14196,3.2596,33.0,5.017657,1.006421,2300.0,3.691814,32.71,-117.03,South
8267,3.8125,49.0,4.473545,1.041005,1314.0,1.738095,33.77,-118.16,East
17445,4.1563,4.0,5.645833,0.985119,915.0,2.723214,34.66,-120.48,West
14265,1.9425,36.0,4.002817,1.033803,1418.0,3.994366,32.69,-117.11,South
2271,3.5542,43.0,6.268421,1.134211,874.0,2.300000,36.78,-119.80,North


In [ ]:
# Identificacao das colunas por tipo
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# Pipeline para variaveis numericas
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para variaveis categoricas
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Criacao do ColumnTransformer unificado
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
# Ajuste e transformaçao nos dados de treino para evitar vazamento de dados
X_train_processed = preprocessor.fit_transform(X_train)

# Apenas transformaçao nos dados de teste utilizando os parametros aprendidos no treino
X_test_processed = preprocessor.transform(X_test)

# Visualizacao das dimensoes finais do dataset processado
print(f'Formato original do treino: {X_train.shape}')
print(f'Formato apos o ColumnTransformer: {X_train_processed.shape}')

Formato original do treino: (16512, 9)
Formato apos o ColumnTransformer: (16512, 12)


# Engenharia de Atributos, Modelagem, Sintonia de Hiperparâmetros e Exportação

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# Indices das colunas necessarias para a engenharia de atributos
rooms_ix, bedrooms_ix, population_ix, households_ix = 2, 3, 4, 5

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Geracao de novas colunas numéricas baseadas em proporcoes
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [ ]:
# Reestruturando o pipeline numerico para incluir a engenharia de atributos
num_pipeline_extended = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('attribs_adder', CombinedAttributesAdder()),
    ('scaler', StandardScaler())
])

# Atualizando o ColumnTransformer com o pipeline numerico estendido
preprocessor_extended = ColumnTransformer([
    ('num', num_pipeline_extended, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Pipeline completo encapsulando preprocessamento e o modelo preditivo
full_pipeline = Pipeline([
    ('preparation', preprocessor_extended),
    ('model', RandomForestRegressor(random_state=42))
])

# Definicao da grade de hiperparametros para busca exaustiva via validacao cruzada
param_grid = [
    {
        'model__n_estimators': [10, 50],
        'model__max_features': [4, 8]
    }
]

# Busca com validacao cruzada de 3 dobras (folds)
grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preparation',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('attribs_adder',
                                                                                          CombinedAttributesAdder()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['MedInc',
                                                                          'HouseAge',
                                                                          'AveRooms',
                                                                          'AveBedrms',
                                                                          'Population',
                                                                          'AveOccup',
                                                                          'Latitude',
                                                                          'Longitude']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore',
                                                                                                        sparse_output=False))]),
                                                                         ['region'])])),
                                       ('model',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid=[{'model__max_features': [4, 8],
                          'model__n_estimators': [10, 50]}],
             return_train_score=True, scoring='neg_mean_squared_error')

In [ ]:
from sklearn.metrics import root_mean_squared_error
import joblib

# Obtencao do melhor estimador encontrado pelo GridSearchCV
best_pipeline = grid_search.best_estimator_

# Predicoes no conjunto de teste usando o pipeline unificado ajustado
y_pred = best_pipeline.predict(X_test)

# Calculo da metrica de desempenho final (RMSE)
rmse = root_mean_squared_error(y_test, y_pred)
print(f'RMSE Final no Conjunto de Teste: {rmse:.4f}')
print(f'Melhores parâmetros encontrados: {grid_search.best_params_}')

# Exportacao do modelo completo pronto para producao
joblib.dump(best_pipeline, 'california_housing_pipeline.joblib')
print('Pipeline exportado com sucesso para o arquivo "california_housing_pipeline.joblib".')

RMSE Final no Conjunto de Teste: 0.5135
Melhores parâmetros encontrados: {'model__max_features': 8, 'model__n_estimators': 50}
Pipeline exportado com sucesso para o arquivo "california_housing_pipeline.joblib".


## Comparação de Modelos Preditivos

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

# Definir uma lista de modelos a serem comparados
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(random_state=42, n_estimators=50, max_features=8), # Usar os melhores parametros encontrados
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Treinando e avaliando: {name}...")

    # Criar um pipeline completo para cada modelo
    current_full_pipeline = Pipeline([
        ('preparation', preprocessor_extended), # Usa o preprocessor estendido com engenharia de atributos
        ('model', model)
    ])

    # Treinar o pipeline
    current_full_pipeline.fit(X_train, y_train)

    # Fazer previsões no conjunto de teste
    y_pred_current = current_full_pipeline.predict(X_test)

    # Calcular o RMSE
    rmse_current = root_mean_squared_error(y_test, y_pred_current)
    results[name] = rmse_current

    print(f'RMSE para {name}: {rmse_current:.4f}\n')

print("\n--- Comparação Final de RMSE ---")
for name, rmse_val in results.items():
    print(f'{name}: {rmse_val:.4f}')

Treinando e avaliando: Linear Regression...
RMSE para Linear Regression: 0.6737

Treinando e avaliando: Decision Tree Regressor...
RMSE para Decision Tree Regressor: 0.7146

Treinando e avaliando: Random Forest Regressor...
RMSE para Random Forest Regressor: 0.5135

Treinando e avaliando: Gradient Boosting Regressor...
RMSE para Gradient Boosting Regressor: 0.5386


--- Comparação Final de RMSE ---
Linear Regression: 0.6737
Decision Tree Regressor: 0.7146
Random Forest Regressor: 0.5135
Gradient Boosting Regressor: 0.5386
